# FLUX.2 LoRA Eğitimi — Ostris AI-Toolkit

Karakter veya stil LoRA eğitimi. 96GB VRAM ile bf16 full precision.

## Dataset Hazırlık
- Karakter: 15-30 görsel (farklı açı, ifade, ışık, kıyafet)
- Stil: 20-50 görsel (aynı stilde farklı konular)
- Çözünürlük: 1024x1024+
- Her görsele caption (.txt dosyası, aynı isimle)

## Kullanım
A: Kurulum → B: Dataset yükle → C: Config ayarla → D: Eğitim başlat → E: LoRA indir

---
# A) Kurulum

In [ ]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# AI-Toolkit kurulumu
TOOLKIT_DIR = '/content/ai-toolkit'
if not os.path.exists(TOOLKIT_DIR):
    print('\U0001f4e6 AI-Toolkit indiriliyor...')
    !git clone --depth 1 https://github.com/ostris/ai-toolkit.git {TOOLKIT_DIR}
    !pip install -q -r {TOOLKIT_DIR}/requirements.txt
    !pip install -q peft accelerate transformers
else:
    print('\u2705 AI-Toolkit mevcut')

print('\u2705 Kurulum tamam')

---
# B) Dataset Yükle

Zip dosyası yükle. İçeriği:
```
dataset/
├── image1.jpg
├── image1.txt    (caption)
├── image2.png
├── image2.txt
└── ...
```
Her görselin yanında aynı isimle .txt caption dosyası olmalı.

In [ ]:
import zipfile
from google.colab import files

DATASET_DIR = '/content/dataset'

print('Dataset zip dosyasını yükle:')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Zip aç
os.makedirs(DATASET_DIR, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)

# İç içe klasör varsa düzelt
subdirs = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(f'{DATASET_DIR}/{d}')]
if len(subdirs) == 1 and not any(f.endswith(('.jpg', '.png', '.webp')) for f in os.listdir(DATASET_DIR)):
    inner = f'{DATASET_DIR}/{subdirs[0]}'
    for f in os.listdir(inner):
        os.rename(f'{inner}/{f}', f'{DATASET_DIR}/{f}')
    os.rmdir(inner)

# Sayım
images = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
captions = [f for f in os.listdir(DATASET_DIR) if f.endswith('.txt')]
print(f'\u2705 {len(images)} görsel, {len(captions)} caption')
if len(images) != len(captions):
    print(f'\u26a0\ufe0f Uyuşmazlık! Her görselin .txt caption\'ı olmalı.')

---
# C) Eğitim Config

In [ ]:
import yaml

# ╔══════════════════════════════════════════════════════════════╗
# ║  EĞİTİM CONFIG                                            ║
# ╚══════════════════════════════════════════════════════════════╝
LORA_NAME       = 'my_lora'          # Çıktı dosya adı
TRIGGER_WORD    = 'ohk_style'        # Trigger word (prompt'ta kullanılacak)
MODEL           = 'flux2_dev'        # 'flux2_dev' | 'flux2_klein'
STEPS           = 3000               # 2000-5000 arası
LEARNING_RATE   = 1e-4
LORA_RANK       = 32                 # 16-32 önerilen
BATCH_SIZE      = 4                  # 96GB VRAM ile 4-8
RESOLUTION      = 1024              # 1024 veya 512
SAVE_EVERY      = 500               # Her N step'te checkpoint kaydet

# Model repo
MODEL_REPO = {
    'flux2_dev': 'black-forest-labs/FLUX.2-dev',
    'flux2_klein': 'black-forest-labs/FLUX.2-klein-9b',
}[MODEL]

OUTPUT_DIR = f'/content/output/{LORA_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# YAML config oluştur
config = {
    'job': 'extension',
    'config': {
        'name': LORA_NAME,
        'process': [{
            'type': 'sd_trainer',
            'training_folder': OUTPUT_DIR,
            'device': 'cuda:0',
            'trigger_word': TRIGGER_WORD,
            'network': {
                'type': 'lora',
                'linear': LORA_RANK,
                'linear_alpha': LORA_RANK,
            },
            'save': {
                'dtype': 'float16',
                'save_every': SAVE_EVERY,
            },
            'datasets': [{
                'folder_path': DATASET_DIR,
                'caption_ext': 'txt',
                'caption_dropout_rate': 0.05,
                'resolution': RESOLUTION,
                'batch_size': BATCH_SIZE,
            }],
            'train': {
                'batch_size': BATCH_SIZE,
                'steps': STEPS,
                'lr': LEARNING_RATE,
                'dtype': 'bf16',
            },
            'model': {
                'name_or_path': MODEL_REPO,
                'is_flux': True,
            },
        }],
    },
}

CONFIG_PATH = '/content/train_config.yaml'
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'\u2705 Config hazır: {CONFIG_PATH}')
print(f'   Model:     {MODEL} ({MODEL_REPO})')
print(f'   Steps:     {STEPS}')
print(f'   LR:        {LEARNING_RATE}')
print(f'   Rank:      {LORA_RANK}')
print(f'   Batch:     {BATCH_SIZE}')
print(f'   Trigger:   {TRIGGER_WORD}')
print(f'   Dataset:   {len(images)} images')
print(f'   Output:    {OUTPUT_DIR}')

---
# D) Eğitim Başlat

In [ ]:
import subprocess
import time

print(f'\U0001f680 Eğitim başlıyor: {LORA_NAME}')
print(f'   Tahmini süre: ~{STEPS // 50} dakika (96GB VRAM)\n')

t0 = time.time()
result = subprocess.run(
    ['python', 'run.py', CONFIG_PATH],
    cwd=TOOLKIT_DIR,
    capture_output=False,
)

elapsed = (time.time() - t0) / 60
if result.returncode == 0:
    print(f'\n\u2705 Eğitim tamamlandı! ({elapsed:.1f} dakika)')
else:
    print(f'\n\u274c Eğitim başarısız (exit: {result.returncode})')

---
# E) LoRA İndir

In [ ]:
import glob
from google.colab import files

# Son checkpoint'i bul
lora_files = sorted(glob.glob(f'{OUTPUT_DIR}/*.safetensors'))

if lora_files:
    print(f'\u2705 LoRA dosyaları ({len(lora_files)}):')
    for f in lora_files:
        size_mb = os.path.getsize(f) / 1024**2
        print(f'   {os.path.basename(f)} ({size_mb:.1f} MB)')

    # Son dosyayı indir
    final = lora_files[-1]
    print(f'\n\U0001f4e6 İndiriliyor: {os.path.basename(final)}')
    files.download(final)
    print(f'\n\U0001f4cb Kullanım:')
    print(f'   ComfyUI/models/loras/ klasörüne koy')
    print(f'   Prompt\'ta "{TRIGGER_WORD}" kullan')
else:
    print('\u274c LoRA dosyası bulunamadı!')
    print(f'   Kontrol et: {OUTPUT_DIR}')